In [1]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
from concurrent.futures import ThreadPoolExecutor
import time
import gc
from tqdm.auto import tqdm

def build_search_corpus(df):
    return (df['clean_name'].fillna('') + " " + 
            df['Trans_name'].fillna('') + " " + 
            df['norm_address'].fillna('')).str.lower()

# ==============================================================================
# 1. LOAD DATA FOR MAXIMUM RECALL BLOCKING
# ==============================================================================
print("1. Loading Data for Maximum Recall Blocking...")
cols = ['entity_id', 'clean_name', 'Trans_name', 'norm_address', 'country']
s1 = pd.read_csv("../ML2026/cleaned_dataset/source1.tsv", sep='\t', usecols=cols).fillna('')
target_df = pd.concat([
    pd.read_csv("../ML2026/cleaned_dataset/source2.tsv", sep='\t', usecols=cols).fillna(''),
    pd.read_csv("../ML2026/cleaned_dataset/source3.tsv", sep='\t', usecols=cols).fillna('')
], ignore_index=True)

s1_ids = s1['entity_id'].values
target_ids = target_df['entity_id'].values

# ==============================================================================
# 2. PASS 1: STABLE & OPTIMIZED TF-IDF MATCHING
# ==============================================================================
print("\n2. Vectorizing with Memory-Capped Relaxed Constraints...")
start_vec = time.time()
vectorizer = TfidfVectorizer(
    analyzer='char_wb', 
    ngram_range=(3, 3), 
    min_df=3,                 
    max_df=0.02,              
    max_features=1_000_000,   
    dtype=np.float32
)

target_corpus = build_search_corpus(target_df)
s1_corpus = build_search_corpus(s1)

vectorizer.fit(target_corpus)
target_matrix = vectorizer.transform(target_corpus)
s1_matrix = vectorizer.transform(s1_corpus)

del target_corpus, s1_corpus
gc.collect()

target_matrix_T = target_matrix.transpose()
print(f"Vectorization complete in {time.time() - start_vec:.2f} seconds.")


1. Loading Data for Maximum Recall Blocking...


KeyboardInterrupt: 

In [ ]:
print("Executing Stable Sparse Dot Product (Capped at 48 Threads)...")
start_math = time.time()

# Chunk size slightly increased for stable C++ batching
chunk_size = 10000
tfidf_s1_matches = []
tfidf_tgt_matches = []

chunks = range(0, s1_matrix.shape[0], chunk_size)

for i in tqdm(chunks, desc="TF-IDF Dot Product"):
    end = min(i + chunk_size, s1_matrix.shape[0])
    
    # 1. Main thread safely slices the matrix (Prevents Segfaults)
    s1_chunk = s1_matrix[i:end]
    
    # 2. C++ backend runs the math on 48 threads (Prevents Cache Thrashing)
    chunk_matches = sp_matmul_topn(
        s1_chunk, 
        target_matrix_T, 
        top_n=35,             
        threshold=0.40,       
        n_threads=80          # <-- The optimized sweet spot for 192 vCPU instances
    )
    
    non_zeros = chunk_matches.nonzero()
    tfidf_s1_matches.extend(s1_ids[non_zeros[0] + i])
    tfidf_tgt_matches.extend(target_ids[non_zeros[1]])

del s1_matrix, target_matrix, target_matrix_T, s1_chunk
gc.collect()

print("Consolidating TF-IDF Pairs into Hash Set...")
all_pairs = set(zip(tfidf_s1_matches, tfidf_tgt_matches))

del tfidf_s1_matches, tfidf_tgt_matches
gc.collect()
print(f"TF-IDF Dot Product complete in {time.time() - start_math:.2f} seconds.")

# ==============================================================================
# 3. PASS 2: EXACT ADDRESS BLOCKING
# ==============================================================================
print("\n3. Executing Exact Address Blocking...")
valid_s1 = s1[s1['norm_address'] != ''][['entity_id', 'norm_address']]
valid_tgt = target_df[target_df['norm_address'] != ''][['entity_id', 'norm_address']]

addr_matches = valid_s1.merge(valid_tgt, on='norm_address')
addr_pairs = set(zip(addr_matches['entity_id_x'], addr_matches['entity_id_y']))

print(f"Found {len(addr_pairs):,} address-based pairs.")
all_pairs.update(addr_pairs)

del valid_s1, valid_tgt, addr_matches, addr_pairs, s1, target_df
gc.collect()

total_generated = len(all_pairs)
print(f"Total Unique Candidates Generated: {total_generated:,}")

# ==============================================================================
# 4. INSTANT BLOCKING RECALL EVALUATION
# ==============================================================================
print("\n4. Loading Ground Truth for Recall Evaluation...")
gt = pd.read_csv("../ML2026/student_resource/dataset/train/train_ground_truth.tsv", sep="\t")
gt['matched_entity_ids'] = gt['matched_entity_ids'].fillna('').astype(str).str.split(',')
gt_exploded = gt.explode('matched_entity_ids')

gt_exploded['source1_entity_id'] = gt_exploded['source1_entity_id'].astype(str).str.strip()
gt_exploded['matched_entity_ids'] = gt_exploded['matched_entity_ids'].astype(str).str.strip()
gt_exploded = gt_exploded[gt_exploded['matched_entity_ids'] != '']

actual_true_pairs = set(zip(gt_exploded['source1_entity_id'], gt_exploded['matched_entity_ids']))
total_actual = len(actual_true_pairs)

print("Calculating Intersection...")
captured_true_pairs = actual_true_pairs.intersection(all_pairs)
total_captured = len(captured_true_pairs)

blocking_recall = total_captured / total_actual if total_actual > 0 else 0
blocking_precision = total_captured / total_generated if total_generated > 0 else 0

print("\n" + "="*45)
print("BLOCKING PHASE METRICS (RELAXED RULES)")
print("="*45)
print(f"True Matches Captured : {total_captured:,} / {total_actual:,}")
print(f"Missed Matches (FN)   : {total_actual - total_captured:,}")
print(f"Blocking Recall       : {blocking_recall:.4f} ({blocking_recall*100:.2f}%)")
print(f"Blocking Precision    : {blocking_precision:.4f} ({blocking_precision*100:.2f}%)")
print("="*45)

# ==============================================================================
# 5. EXPORT HIGH-RECALL DATASET
# ==============================================================================
print("\n5. Writing 'high_recall_candidate_pairs.csv' to disk...")
df_out = pd.DataFrame(list(all_pairs), columns=['s1_entity_id', 'target_entity_id'])
df_out.to_csv("high_recall_candidate_pairs.csv", index=False)
print("Export complete. Dataset is ready for XGBoost Feature Engineering.")

In [ ]:
!pip install sparse_dot_topn


In [ ]:
import joblib

joblib.dump(vectorizer, 'vectorizer.pkl')

In [ ]:
joblib.dump(s1_matrix, 's1_matrix.pkl')

In [ ]:
joblib.dump(target_matrix, 'target_matrix.pkl')

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
import time
import gc
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings('ignore')

print("1. Loading and Preprocessing Data...")
cols = ['entity_id', 'clean_name', 'Trans_name', 'norm_address', 'country']

# Load and fill NaNs (Crucial for Pandas Hash Merging)
s1 = pd.read_csv("../ML2026/cleaned_dataset/source1.tsv", sep='\t', usecols=cols).fillna('')
target_df = pd.concat([
    pd.read_csv("../ML2026/cleaned_dataset/source2.tsv", sep='\t', usecols=cols).fillna(''),
    pd.read_csv("../ML2026/cleaned_dataset/source3.tsv", sep='\t', usecols=cols).fillna('')
], ignore_index=True)

# Treat empty countries as a distinct group to avoid dropping rows during merge
s1['country'] = s1['country'].replace('', 'UNKNOWN_COUNTRY')
target_df['country'] = target_df['country'].replace('', 'UNKNOWN_COUNTRY')

candidate_blocks = []

# ==============================================================================
# PHASE A: O(1) EXACT MATCHING ("Sure Matches" -> Confidence 1.0)
# ==============================================================================
print("\n2. Executing O(1) Hash Joins for Exact Matches...")
start_exact = time.time()

# 1 & 2. Exact Clean Name + Country Match
valid_s1 = s1[s1['clean_name'] != '']
valid_tgt = target_df[target_df['clean_name'] != '']
m1 = valid_s1.merge(valid_tgt, on=['country', 'clean_name'])[['entity_id_x', 'entity_id_y']]
m1.columns = ['s1_entity_id', 'target_entity_id']
m1['match_type'] = 'exact_name_country'
m1['confidence_score'] = 1.0
candidate_blocks.append(m1)
print(f" -> Exact Clean Name Matches: {len(m1):,}")

# 1 & 4. Exact Transliterated Name + Country Match
valid_s1 = s1[s1['Trans_name'] != '']
valid_tgt = target_df[target_df['Trans_name'] != '']
m2 = valid_s1.merge(valid_tgt, on=['country', 'Trans_name'])[['entity_id_x', 'entity_id_y']]
m2.columns = ['s1_entity_id', 'target_entity_id']
m2['match_type'] = 'exact_trans_country'
m2['confidence_score'] = 1.0
candidate_blocks.append(m2)
print(f" -> Exact Transliterated Matches: {len(m2):,}")

# 1 & 3. Exact Address Match + Country Match
valid_s1 = s1[s1['norm_address'] != '']
valid_tgt = target_df[target_df['norm_address'] != '']
m3 = valid_s1.merge(valid_tgt, on=['country', 'norm_address'])[['entity_id_x', 'entity_id_y']]
m3.columns = ['s1_entity_id', 'target_entity_id']
m3['match_type'] = 'exact_address_country'
m3['confidence_score'] = 1.0
candidate_blocks.append(m3)
print(f" -> Exact Address Matches: {len(m3):,}")

del valid_s1, valid_tgt, m1, m2, m3
gc.collect()

print(f"Exact Matching completed in {time.time() - start_exact:.2f} seconds.")

# ==============================================================================
# PHASE B: TF-IDF TOKEN MATCHING ("Likely Candidates" -> Partial Score)
# ==============================================================================
def execute_tfidf_blocking(col_name, match_label, threshold, top_n):
    print(f"\nExecuting TF-IDF Word-Token Blocking for '{col_name}'...")
    start_tf = time.time()
    
    # Using 'word' analyzer makes the matrix highly sparse and prevents CPU locking
    vectorizer = TfidfVectorizer(
        analyzer='word', 
        ngram_range=(1, 2),  # Unigrams and Bigrams
        max_df=0.01,         # Strict cutoff for common words (e.g., "limited", "road")
        dtype=np.float32
    )

    corpus_tgt = target_df[col_name].values
    corpus_s1 = s1[col_name].values

    vectorizer.fit(corpus_tgt)
    mat_tgt = vectorizer.transform(corpus_tgt).transpose().tocsr()
    mat_s1 = vectorizer.transform(corpus_s1)

    s1_ids = s1['entity_id'].values
    tgt_ids = target_df['entity_id'].values

    s1_matches, tgt_matches, scores = [], [], []
    chunk_size = 20000

    chunks = range(0, mat_s1.shape[0], chunk_size)
    for i in tqdm(chunks, desc=f"Dot Product ({col_name})"):
        end = min(i + chunk_size, mat_s1.shape[0])
        
        # 64 threads avoids False Sharing on the CPU cache while maximizing speed
        matches = sp_matmul_topn(
            mat_s1[i:end], mat_tgt, top_n=top_n, threshold=threshold, n_threads=64
        )
        
        non_zeros = matches.nonzero()
        s1_matches.extend(s1_ids[non_zeros[0] + i])
        tgt_matches.extend(tgt_ids[non_zeros[1]])
        scores.extend(matches.data)

    df_tf = pd.DataFrame({
        's1_entity_id': s1_matches,
        'target_entity_id': tgt_matches,
        'match_type': match_label,
        'confidence_score': np.round(scores, 4)
    })
    
    print(f" -> Found {len(df_tf):,} '{match_label}' candidates in {time.time() - start_tf:.2f} sec.")
    return df_tf

# 6. TF-IDF Name Token Matching
df_tf_name = execute_tfidf_blocking(
    col_name='clean_name', match_label='tfidf_name_token', threshold=0.50, top_n=20
)
candidate_blocks.append(df_tf_name)

# 5. TF-IDF Address Token Matching
df_tf_addr = execute_tfidf_blocking(
    col_name='norm_address', match_label='tfidf_address_token', threshold=0.60, top_n=20
)
candidate_blocks.append(df_tf_addr)

del df_tf_name, df_tf_addr
gc.collect()

# ==============================================================================
# PHASE C: CONSOLIDATION & DEDUPLICATION
# ==============================================================================
print("\n3. Consolidating and Deduplicating Candidates...")
final_candidates = pd.concat(candidate_blocks, ignore_index=True)
del candidate_blocks
gc.collect()

# Sort by confidence score so that when we drop duplicates, we KEEP the "Sure Matches" (1.0)
final_candidates = final_candidates.sort_values('confidence_score', ascending=False)
final_candidates = final_candidates.drop_duplicates(subset=['s1_entity_id', 'target_entity_id'], keep='first')
final_candidates.reset_index(drop=True, inplace=True)

total_generated = len(final_candidates)
print(f"Total Unique Candidates Generated: {total_generated:,}")

final_candidates.to_csv("labeled_candidate_pairs.csv", index=False)
print("Exported to 'labeled_candidate_pairs.csv'.")

# ==============================================================================
# PHASE D: BLOCKING RECALL EVALUATION
# ==============================================================================
print("\n4. Evaluating Blocking Recall...")
gt = pd.read_csv("../ML2026/student_resource/dataset/train/train_ground_truth.tsv", sep="\t")
gt['matched_entity_ids'] = gt['matched_entity_ids'].fillna('').astype(str).str.split(',')
gt_exploded = gt.explode('matched_entity_ids')

gt_exploded['source1_entity_id'] = gt_exploded['source1_entity_id'].astype(str).str.strip()
gt_exploded['matched_entity_ids'] = gt_exploded['matched_entity_ids'].astype(str).str.strip()
gt_exploded = gt_exploded[gt_exploded['matched_entity_ids'] != '']

actual_true_pairs = set(zip(gt_exploded['source1_entity_id'], gt_exploded['matched_entity_ids']))
total_actual = len(actual_true_pairs)

generated_pairs_set = set(zip(final_candidates['s1_entity_id'], final_candidates['target_entity_id']))

captured_true_pairs = actual_true_pairs.intersection(generated_pairs_set)
total_captured = len(captured_true_pairs)

blocking_recall = total_captured / total_actual if total_actual > 0 else 0
blocking_precision = total_captured / total_generated if total_generated > 0 else 0

print("\n" + "="*45)
print("HIERARCHICAL BLOCKING METRICS")
print("="*45)
print(f"True Matches Captured : {total_captured:,} / {total_actual:,}")
print(f"Missed Matches (FN)   : {total_actual - total_captured:,}")
print(f"Blocking Recall       : {blocking_recall:.4f} ({blocking_recall*100:.2f}%)")
print(f"Blocking Precision    : {blocking_precision:.4f} ({blocking_precision*100:.2f}%)")
print("="*45)

1. Loading and Preprocessing Data...

2. Executing O(1) Hash Joins for Exact Matches...
 -> Exact Clean Name Matches: 46,195,506
 -> Exact Transliterated Matches: 46,052,156
 -> Exact Address Matches: 1,007,476
Exact Matching completed in 83.32 seconds.

Executing TF-IDF Word-Token Blocking for 'clean_name'...


Dot Product (clean_name):   0%|          | 0/111 [00:00<?, ?it/s]

 -> Found 29,635,756 'tfidf_name_token' candidates in 409.13 sec.

Executing TF-IDF Word-Token Blocking for 'norm_address'...


Dot Product (norm_address):   0%|          | 0/111 [00:00<?, ?it/s]

 -> Found 9,259,127 'tfidf_address_token' candidates in 900.24 sec.

3. Consolidating and Deduplicating Candidates...
Total Unique Candidates Generated: 101,208,922
Exported to 'labeled_candidate_pairs.csv'.

4. Evaluating Blocking Recall...

HIERARCHICAL BLOCKING METRICS
True Matches Captured : 6,259,009 / 7,638,365
Missed Matches (FN)   : 1,379,356
Blocking Recall       : 0.8194 (81.94%)
Blocking Precision    : 0.0618 (6.18%)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
import time
import gc
import re
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings('ignore')

print("1. Loading and Preprocessing Data...")
cols = ['entity_id', 'clean_name', 'Trans_name', 'norm_address', 'country']

s1 = pd.read_csv("../ML2026/cleaned_dataset/source1.tsv", sep='\t', usecols=cols).fillna('')
target_df = pd.concat([
    pd.read_csv("../ML2026/cleaned_dataset/source2.tsv", sep='\t', usecols=cols).fillna(''),
    pd.read_csv("../ML2026/cleaned_dataset/source3.tsv", sep='\t', usecols=cols).fillna('')
], ignore_index=True)

s1['country'] = s1['country'].replace('', 'UNKNOWN_COUNTRY')
target_df['country'] = target_df['country'].replace('', 'UNKNOWN_COUNTRY')

# ==============================================================================
# PHASE A: FEATURE ENGINEERING FOR ADVANCED BLOCKING
# ==============================================================================
print("Generating Advanced Blocking Features (Acronyms, Digits, Sorted Tokens)...")

def sort_string(text):
    return " ".join(sorted(str(text).split()))

def extract_digits(text):
    digits = re.sub(r'\D+', '', str(text))
    return digits if len(digits) >= 6 else ''

def make_acronym(text):
    words = str(text).split()
    return "".join([w[0] for w in words]) if len(words) > 1 else ''

s1['sorted_name'] = s1['clean_name'].apply(sort_string)
target_df['sorted_name'] = target_df['clean_name'].apply(sort_string)

s1['addr_digits'] = s1['norm_address'].apply(extract_digits)
target_df['addr_digits'] = target_df['norm_address'].apply(extract_digits)

s1['acronym'] = s1['clean_name'].apply(make_acronym)
target_df['acronym'] = target_df['clean_name'].apply(make_acronym)

candidate_blocks = []

# ==============================================================================
# PHASE B: O(1) EXACT MATCHING ("Sure Matches" & "High Probability")
# ==============================================================================
print("\n2. Executing O(1) Hash Joins for Exact Matches...")
start_exact = time.time()

def hash_join(df1, df2, join_keys, match_label, confidence):
    valid_1 = df1[(df1[join_keys[-1]] != '') & (df1[join_keys[-1]].notna())]
    valid_2 = df2[(df2[join_keys[-1]] != '') & (df2[join_keys[-1]].notna())]
    m = valid_1.merge(valid_2, on=join_keys)[['entity_id_x', 'entity_id_y']]
    m.columns = ['s1_entity_id', 'target_entity_id']
    m['match_type'] = match_label
    m['confidence_score'] = confidence
    print(f" -> {match_label}: {len(m):,}")
    return m

# 1. Exact Clean Name
candidate_blocks.append(hash_join(s1, target_df, ['country', 'clean_name'], 'exact_name_country', 1.0))
# 2. Exact Transliterated Name
candidate_blocks.append(hash_join(s1, target_df, ['country', 'Trans_name'], 'exact_trans_country', 1.0))
# 3. Exact Address Match
candidate_blocks.append(hash_join(s1, target_df, ['country', 'norm_address'], 'exact_address_country', 1.0))
# 4. Sorted Name Match (Catches Word Reordering)
candidate_blocks.append(hash_join(s1, target_df, ['country', 'sorted_name'], 'sorted_name_match', 0.9))
# 5. Address Digits Match (Catches PIN + Plot No. combos despite typos)
candidate_blocks.append(hash_join(s1, target_df, ['country', 'addr_digits'], 'address_digits_match', 0.8))

# 6. Acronym Matches (Cross-Column: S1 Acronym == Target Name & Vice Versa)
valid_s1_acr = s1[s1['acronym'] != '']
valid_tgt_name = target_df[target_df['clean_name'] != '']
m_acr1 = valid_s1_acr.merge(valid_tgt_name, left_on=['country', 'acronym'], right_on=['country', 'clean_name'])[['entity_id_x', 'entity_id_y']]
m_acr1.columns = ['s1_entity_id', 'target_entity_id']
m_acr1['match_type'] = 'acronym_match'
m_acr1['confidence_score'] = 0.7
candidate_blocks.append(m_acr1)
print(f" -> Acronym Match (S1 Acr -> Tgt Name): {len(m_acr1):,}")

valid_s1_name = s1[s1['clean_name'] != '']
valid_tgt_acr = target_df[target_df['acronym'] != '']
m_acr2 = valid_s1_name.merge(valid_tgt_acr, left_on=['country', 'clean_name'], right_on=['country', 'acronym'])[['entity_id_x', 'entity_id_y']]
m_acr2.columns = ['s1_entity_id', 'target_entity_id']
m_acr2['match_type'] = 'acronym_match'
m_acr2['confidence_score'] = 0.7
candidate_blocks.append(m_acr2)
print(f" -> Acronym Match (S1 Name -> Tgt Acr): {len(m_acr2):,}")

del valid_s1_acr, valid_tgt_name, valid_s1_name, valid_tgt_acr, m_acr1, m_acr2
gc.collect()

print(f"Exact Hash Matching completed in {time.time() - start_exact:.2f} seconds.")

# ==============================================================================
# PHASE C: TF-IDF TOKEN & N-GRAM MATCHING ("Likely Candidates")
# ==============================================================================
def execute_tfidf_blocking(col_name, match_label, threshold, top_n, analyzer='word', ngram_range=(1, 2), max_features=None):
    print(f"\nExecuting TF-IDF Blocking ({analyzer}) for '{col_name}'...")
    start_tf = time.time()
    
    vectorizer = TfidfVectorizer(
        analyzer=analyzer, 
        ngram_range=ngram_range,
        max_df=0.01 if analyzer == 'word' else 0.05,
        max_features=max_features, # Critical for character N-Grams to prevent segfaults
        dtype=np.float32
    )

    corpus_tgt = target_df[col_name].values
    corpus_s1 = s1[col_name].values

    vectorizer.fit(corpus_tgt)
    mat_tgt = vectorizer.transform(corpus_tgt).transpose().tocsr()
    mat_s1 = vectorizer.transform(corpus_s1)

    s1_ids = s1['entity_id'].values
    tgt_ids = target_df['entity_id'].values

    s1_matches, tgt_matches, scores = [], [], []
    chunk_size = 20000

    chunks = range(0, mat_s1.shape[0], chunk_size)
    for i in tqdm(chunks, desc=f"Dot Product ({match_label})"):
        end = min(i + chunk_size, mat_s1.shape[0])
        
        matches = sp_matmul_topn(
            mat_s1[i:end], mat_tgt, top_n=top_n, threshold=threshold, n_threads=64
        )
        
        non_zeros = matches.nonzero()
        s1_matches.extend(s1_ids[non_zeros[0] + i])
        tgt_matches.extend(tgt_ids[non_zeros[1]])
        scores.extend(matches.data)

    df_tf = pd.DataFrame({
        's1_entity_id': s1_matches,
        'target_entity_id': tgt_matches,
        'match_type': match_label,
        'confidence_score': np.round(scores, 4)
    })
    
    print(f" -> Found {len(df_tf):,} '{match_label}' candidates in {time.time() - start_tf:.2f} sec.")
    del mat_tgt, mat_s1, corpus_tgt, corpus_s1
    gc.collect()
    return df_tf

# 7. TF-IDF Name Token Matching (Word Level)
candidate_blocks.append(execute_tfidf_blocking(
    col_name='clean_name', match_label='tfidf_name_word', threshold=0.50, top_n=20, analyzer='word'
))

# 8. TF-IDF Address Token Matching (Word Level)
candidate_blocks.append(execute_tfidf_blocking(
    col_name='norm_address', match_label='tfidf_addr_word', threshold=0.60, top_n=20, analyzer='word'
))

# 9. TF-IDF Name N-Gram Matching (Catches deep character typos like "McDonalds" vs "MacDonalds")
candidate_blocks.append(execute_tfidf_blocking(
    col_name='clean_name', match_label='tfidf_name_char', threshold=0.45, top_n=30, 
    analyzer='char_wb', ngram_range=(3, 4), max_features=250_000
))

# ==============================================================================
# PHASE D: CONSOLIDATION & DEDUPLICATION
# ==============================================================================
print("\n3. Consolidating and Deduplicating Candidates...")
final_candidates = pd.concat(candidate_blocks, ignore_index=True)
del candidate_blocks, s1, target_df
gc.collect()

# Sort by confidence score so that when we drop duplicates, we KEEP the higher confidence labels
final_candidates = final_candidates.sort_values('confidence_score', ascending=False)
final_candidates = final_candidates.drop_duplicates(subset=['s1_entity_id', 'target_entity_id'], keep='first')
final_candidates.reset_index(drop=True, inplace=True)

total_generated = len(final_candidates)
print(f"Total Unique Candidates Generated: {total_generated:,}")

final_candidates.to_csv("labeled_candidate_pairs.csv", index=False)
print("Exported to 'labeled_candidate_pairs.csv'.")

# ==============================================================================
# PHASE E: BLOCKING RECALL EVALUATION
# ==============================================================================
print("\n4. Evaluating Blocking Recall...")
gt = pd.read_csv("../ML2026/student_resource/dataset/train/train_ground_truth.tsv", sep="\t")
gt['matched_entity_ids'] = gt['matched_entity_ids'].fillna('').astype(str).str.split(',')
gt_exploded = gt.explode('matched_entity_ids')

gt_exploded['source1_entity_id'] = gt_exploded['source1_entity_id'].astype(str).str.strip()
gt_exploded['matched_entity_ids'] = gt_exploded['matched_entity_ids'].astype(str).str.strip()
gt_exploded = gt_exploded[gt_exploded['matched_entity_ids'] != '']

actual_true_pairs = set(zip(gt_exploded['source1_entity_id'], gt_exploded['matched_entity_ids']))
total_actual = len(actual_true_pairs)

generated_pairs_set = set(zip(final_candidates['s1_entity_id'], final_candidates['target_entity_id']))

captured_true_pairs = actual_true_pairs.intersection(generated_pairs_set)
total_captured = len(captured_true_pairs)

blocking_recall = total_captured / total_actual if total_actual > 0 else 0
blocking_precision = total_captured / total_generated if total_generated > 0 else 0

print("\n" + "="*45)
print("HIERARCHICAL BLOCKING METRICS (MAX RECALL)")
print("="*45)
print(f"True Matches Captured : {total_captured:,} / {total_actual:,}")
print(f"Missed Matches (FN)   : {total_actual - total_captured:,}")
print(f"Blocking Recall       : {blocking_recall:.4f} ({blocking_recall*100:.2f}%)")
print(f"Blocking Precision    : {blocking_precision:.4f} ({blocking_precision*100:.2f}%)")
print("="*45)

1. Loading and Preprocessing Data...
Generating Advanced Blocking Features (Acronyms, Digits, Sorted Tokens)...

2. Executing O(1) Hash Joins for Exact Matches...
 -> exact_name_country: 46,195,506
 -> exact_trans_country: 46,052,156
 -> exact_address_country: 1,007,476
 -> sorted_name_match: 47,639,615
 -> address_digits_match: 1,528,549
 -> Acronym Match (S1 Acr -> Tgt Name): 32,243,149
 -> Acronym Match (S1 Name -> Tgt Acr): 1,614,153
Exact Hash Matching completed in 244.28 seconds.

Executing TF-IDF Blocking (word) for 'clean_name'...


Dot Product (tfidf_name_word):   0%|          | 0/111 [00:00<?, ?it/s]

 -> Found 29,635,756 'tfidf_name_word' candidates in 403.42 sec.

Executing TF-IDF Blocking (word) for 'norm_address'...


Dot Product (tfidf_addr_word):   0%|          | 0/111 [00:00<?, ?it/s]

 -> Found 9,259,127 'tfidf_addr_word' candidates in 910.42 sec.

Executing TF-IDF Blocking (char_wb) for 'clean_name'...


Dot Product (tfidf_name_char):   0%|          | 0/111 [00:00<?, ?it/s]